# Customer Segmentation — K-Means Clustering + RFM Scoring

## 1. Setup & Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.pipeline import Pipeline

df = pd.read_csv('/content/drive/MyDrive/Kaggle/E-Commerce Data Set/ecommerce_user_behavior_8000.csv')

# Clean
df_clean = df.copy()
for col in df_clean.select_dtypes(include='number').columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)
df_clean['device_type'].fillna('Unknown', inplace=True)
df_clean['gender'].fillna('Unknown', inplace=True)
df_clean['purchase'] = df_clean['purchase'].fillna(0).astype(int)

print(f'Dataset shape: {df_clean.shape}')
df_clean.describe().round(2)

## 2. RFM-Inspired Feature Engineering

In [ ]:
# Build RFM-style features
df_rfm = df_clean.copy()

# Recency proxy: avg_session_time (more recent/active = higher session time)
# Frequency proxy: pages_viewed + previous_purchases
# Monetary proxy: cart_items (intent to spend)
# Engagement: time_on_site, returning_user

df_rfm['recency_score'] = df_rfm['avg_session_time']  # higher = more active recently
df_rfm['frequency_score'] = df_rfm['pages_viewed'] + df_rfm['previous_purchases']
df_rfm['monetary_score'] = df_rfm['cart_items']
df_rfm['engagement_score'] = df_rfm['time_on_site'] * (1 + df_rfm['returning_user'])
df_rfm['intent_score'] = df_rfm['cart_items'] * (1 - df_rfm['bounce_rate']/100)

clustering_features = [
    'recency_score', 'frequency_score', 'monetary_score',
    'engagement_score', 'intent_score', 'bounce_rate'
]

X_cluster = df_rfm[clustering_features]
print('Clustering features:')
X_cluster.describe().round(2)

## 3. Optimal K Selection — Elbow + Silhouette

In [ ]:
# Preprocess
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(imputer.fit_transform(X_cluster))

inertias = []
silhouettes = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'bo-', lw=2, ms=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xticks(list(k_range))

axes[1].plot(list(k_range), silhouettes, 'rs-', lw=2, ms=8)
optimal_k = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(optimal_k, color='navy', linestyle='--', lw=2, label=f'Optimal k={optimal_k}')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontweight='bold')
axes[1].set_xticks(list(k_range))
axes[1].legend()

plt.tight_layout()
plt.savefig('optimal_k.png', bbox_inches='tight')
plt.show()
print(f'\nOptimal k by Silhouette: {optimal_k}')

## 4. Final K-Means Clustering

In [ ]:
K = 4  # Business-interpretable number of segments

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
df_rfm['cluster'] = km_final.fit_predict(X_scaled)

print(f'Silhouette Score (k={K}): {silhouette_score(X_scaled, df_rfm["cluster"]):.4f}')
print(f'\nCluster sizes:')
print(df_rfm['cluster'].value_counts().sort_index())

## 5. Cluster Profiling

In [ ]:
profile_cols = clustering_features + ['purchase', 'age', 'previous_purchases', 'cart_items']
cluster_profiles = df_rfm.groupby('cluster')[profile_cols].mean().round(2)
cluster_profiles['n_users'] = df_rfm['cluster'].value_counts().sort_index()
cluster_profiles['conversion_rate'] = (cluster_profiles['purchase'] * 100).round(1)

cluster_profiles

In [ ]:
# Name clusters based on profiles
conv_ranked = cluster_profiles['conversion_rate'].rank()
engagement_ranked = cluster_profiles['engagement_score'].rank()
intent_ranked = cluster_profiles['intent_score'].rank()

# Auto-assign persona names
def assign_persona(idx):
    conv = cluster_profiles.loc[idx, 'conversion_rate']
    eng = cluster_profiles.loc[idx, 'engagement_score']
    intent = cluster_profiles.loc[idx, 'intent_score']
    bounce = cluster_profiles.loc[idx, 'bounce_rate']

    if conv > 60 and intent > cluster_profiles['intent_score'].median():
        return 'High-Value Buyers'
    elif eng > cluster_profiles['engagement_score'].median() and conv < 50:
        return 'Engaged Browsers'
    elif bounce > cluster_profiles['bounce_rate'].median() and conv < 50:
        return 'At-Risk / Bouncers'
    else:
        return 'Casual Shoppers'

personas = {idx: assign_persona(idx) for idx in cluster_profiles.index}
df_rfm['persona'] = df_rfm['cluster'].map(personas)

print('Cluster → Persona Mapping:')
for k, v in personas.items():
    print(f'  Cluster {k}: {v}')

## 6. PCA Visualization of Clusters

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

# PCA scatter
for cluster_id in range(K):
    mask = df_rfm['cluster'] == cluster_id
    persona_name = personas[cluster_id]
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=palette[cluster_id], s=15, alpha=0.5,
                    label=f'C{cluster_id}: {persona_name}')

# Cluster centroids in PCA space
centroids_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', s=200, marker='*', zorder=5, label='Centroids')

axes[0].set_title(f'K-Means Clusters (k={K}) — PCA Projection', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend(fontsize=9, loc='best')

# Conversion rate by cluster
conv_by_cluster = df_rfm.groupby('cluster')['purchase'].mean() * 100
conv_by_cluster.index = [f'C{i}\n{personas[i]}' for i in conv_by_cluster.index]
bars = axes[1].bar(conv_by_cluster.index, conv_by_cluster.values, color=palette, edgecolor='white')
axes[1].set_title('Conversion Rate by Customer Segment', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for bar, v in zip(bars, conv_by_cluster.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('cluster_visualization.png', bbox_inches='tight')
plt.show()

## 7. Radar Chart — Cluster Feature Profiles

In [ ]:
radar_features = ['recency_score', 'frequency_score', 'monetary_score',
                  'engagement_score', 'intent_score']

# Normalize per feature to 0–1
radar_df = cluster_profiles[radar_features].copy()
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min())

N = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

labels_radar = ['Recency', 'Frequency', 'Monetary', 'Engagement', 'Intent']

fig, axes = plt.subplots(1, K, figsize=(16, 4), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Profiles — Normalized Radar Charts', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes):
    values = radar_norm.loc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, color=palette[i], lw=2)
    ax.fill(angles, values, color=palette[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_radar, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['25', '50', '75'], fontsize=7)
    ax.set_title(f'C{i}: {personas[i]}', fontweight='bold', pad=15, fontsize=10)

plt.tight_layout()
plt.savefig('radar_profiles.png', bbox_inches='tight')
plt.show()

## 8. Customer Value Scoring

In [ ]:
# Composite customer value score (0-100)
def percentile_rank(series):
    return series.rank(pct=True) * 100

df_rfm['value_score'] = (
    0.25 * percentile_rank(df_rfm['recency_score']) +
    0.25 * percentile_rank(df_rfm['frequency_score']) +
    0.30 * percentile_rank(df_rfm['intent_score']) +
    0.20 * percentile_rank(df_rfm['engagement_score'])
)

df_rfm['value_tier'] = pd.cut(
    df_rfm['value_score'],
    bins=[0, 25, 50, 75, 100],
    labels=['Low', 'Medium', 'High', 'Premium']
)

tier_summary = df_rfm.groupby('value_tier').agg(
    n_users=('user_id','count'),
    conversion_rate=('purchase','mean'),
    avg_cart_items=('cart_items','mean'),
    avg_session_time=('avg_session_time','mean')
).round(3)
tier_summary['conversion_rate'] = (tier_summary['conversion_rate'] * 100).round(1)

print(tier_summary)

## 9. Summary & Marketing Recommendations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cluster size pie
sizes = df_rfm.groupby('cluster').size()
persona_labels = [f'C{i}: {personas[i]}' for i in sizes.index]
axes[0].pie(sizes, labels=persona_labels, colors=palette, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 9})
axes[0].set_title('Customer Segment Distribution', fontweight='bold')

# Value tier bar
tier_colors = ['#90A4AE', '#42A5F5', '#66BB6A', '#FF7043']
tier_conv = df_rfm.groupby('value_tier')['purchase'].mean() * 100
axes[1].bar(tier_conv.index, tier_conv.values, color=tier_colors, edgecolor='white')
axes[1].set_title('Conversion Rate by Value Tier', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for i, v in enumerate(tier_conv.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('segment_summary.png', bbox_inches='tight')
plt.show()

for cluster_id, persona in personas.items():
    conv = cluster_profiles.loc[cluster_id, 'conversion_rate']
    n = cluster_profiles.loc[cluster_id, 'n_users']
    if 'High-Value' in persona:
        action = 'Nurture with loyalty programs and early access offers'
    elif 'Engaged' in persona:
        action = 'Convert with targeted discounts and urgency messaging'
    elif 'At-Risk' in persona:
        action = 'Re-engage with exit-intent popups and retargeting'
    else:
        action = 'Educate and build awareness with top-of-funnel content'
    print(f'C{cluster_id} [{persona}] | Conv={conv:.1f}% | n={int(n):,}')
    print(f'   → {action}')
    print()